# Feature Preparation

One-time setup notebook. Run this **before** `microstructure_demo`, `pipeline_benchmark`, or `bayes_tuning`.

Each section is idempotent: it skips work that is already done. Re-run with `force=True` only when you add new images or want to rebuild a cache.

| Section | Writes | Skip condition |
|---------|--------|----------------|
| §2 Download | `data/temp_images/*.jpg` | images already present |
| §3 CNN extraction | `data/image_cache_<backbone>.npz` | cache file exists |
| §4 Morph extraction | `features/morph_features_c1.npz` | cache file exists |
| §5 Verify | — | always runs |

**Library**: all logic lives in `src/features.py` (`FeaturePipeline`). For CI/CD or script use, import it directly.


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.features import FeaturePipeline
from src.column_sanitizer import sanitize_dataframe

print('Setup complete.')


## §1 — Configuration

In [ ]:
DATA_DIR     = os.path.abspath('../data')
TEMP_DIR     = os.path.abspath('../data/temp_images')
FEATURES_DIR = os.path.abspath('../features')
CREDS_PATH   = os.path.abspath('../credentials.json')
TOKEN_PATH   = os.path.abspath('../token.json')
DATASET      = os.path.abspath('../datasets/metadata_latest.csv')

# Backbones to extract. None = all registered backbones.
# BACKBONES = ['dinov2_vitb14', 'resnet50']
BACKBONES    = None

IMG_SIZE     = 224
BATCH_SIZE   = 16
NUM_WORKERS  = 2

# Google Drive folder column. Set to None to skip download.
FOLDER_COL   = 'augumented_data'
ID_COL       = 'id'

fp = FeaturePipeline(
    data_dir=DATA_DIR,
    temp_dir=TEMP_DIR,
    features_dir=FEATURES_DIR,
    credentials_path=CREDS_PATH,
    token_path=TOKEN_PATH,
)

fp.print_status()


## §2 — Download Images

Reads Google Drive folder links from `FOLDER_COL` and downloads all images to `data/temp_images/`. Skips if images are already present.

Set `FOLDER_COL = None` or skip this cell if you placed images manually.


In [ ]:
df_raw = pd.read_csv(DATASET, header=1)
df_raw = sanitize_dataframe(df_raw)

if FOLDER_COL is None:
    print('FOLDER_COL is None -- skipping download.')
    print(f'Expecting images already in: {TEMP_DIR}')
else:
    image_paths = fp.download_images(
        df=df_raw,
        folder_col=FOLDER_COL,
        id_col=ID_COL,
        # force=True,
    )
    print(f'Images available: {len(image_paths)}')


## §3 — CNN Feature Extraction

Runs each backbone over all images in `data/temp_images/` and saves one `.npz` cache per backbone. Skips any backbone whose cache already exists.

GPU is used automatically if available; falls back to CPU.


In [ ]:
image_paths = fp._list_images()
if not image_paths:
    print(f'No images found in {TEMP_DIR}.')
    print('Run §2 first, or place images manually in that directory.')
else:
    print(f'Found {len(image_paths)} images.')
    written = fp.extract_cnn(
        image_paths=image_paths,
        backbones=BACKBONES,
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        # force=True,
    )
    print(f'\nCaches written/confirmed: {len(written)}')
    import numpy as _np
    for name, path in written.items():
        d = _np.load(str(path), allow_pickle=True)
        print(f'  {name:<22} {d["X"].shape}')


## §4 — Morphological Feature Extraction

Extracts 33 physically-interpretable features per image (phase fractions, grain geometry, GLCM texture, LBP) and caches to `features/morph_features_c1.npz`. CPU-only; ~1–3 s per image.


In [ ]:
image_paths = fp._list_images()
if not image_paths:
    print(f'No images found in {TEMP_DIR} -- skipping morph extraction.')
else:
    morph_path = fp.extract_morph(
        image_paths=image_paths,
        # force=True,
    )
    import numpy as _np
    d = _np.load(str(morph_path), allow_pickle=True)
    print(f'Morph cache: {d["X"].shape} -> {morph_path}')


## §5 — Verify Caches

Run at any time to check what has been built. Downstream notebooks degrade gracefully if a cache is missing, but full feature matrices require all caches to be present.


In [ ]:
fp.print_status()

status = fp.verify()
missing = []
for backbone, info in status["cnn"].items():
    if not info["exists"]:
        missing.append(f'CNN cache: {backbone}')
if not status["morph"]["exists"]:
    missing.append('Morph cache')

if missing:
    print('\nMissing caches:')
    for m in missing:
        print(f'  x {m}')
else:
    print('\nAll caches present -- ready for pipeline/benchmark/bayes_tuning.')
